# SadTalker 宠物网格图生成器 (Google Colab)

本 Notebook 使用 SadTalker 在 Colab 上为宠物生成短视频，并自动转换为多视角网格图，供你的前端交互使用。

## 工作流程
1. 检查 GPU 并安装依赖
2. 克隆 SadTalker 仓库并安装模型依赖
3. 上传宠物照片（可选上传驱动音频）
4. 运行 SadTalker 生成宠物动画视频
5. 从视频中抽帧并拼接成 7×7 网格图
6. 下载网格图，放回本地 `face_demo/outputs/` 目录使用


In [ ]:
# 步骤 1: 检查 GPU
import torch
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 未检测到 GPU，请在 Colab 运行时设置中选择 GPU")


## 步骤 2: 克隆 SadTalker 仓库并安装依赖


In [ ]:
# 克隆 SadTalker 仓库
import os
if not os.path.exists('SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git
    print("✅ SadTalker 仓库已克隆")
else:
    print("✅ SadTalker 仓库已存在")

%cd SadTalker

# 安装依赖
!pip install -q -r requirements.txt
!pip install -q opencv-python-headless imageio imageio-ffmpeg

print("✅ SadTalker 依赖安装完成")


## 步骤 3: 上传宠物照片和（可选）驱动音频


In [ ]:
from google.colab import files
import shutil
import os

os.makedirs('inputs', exist_ok=True)

print("📤 请上传宠物照片（建议正脸、清晰、无遮挡）")
img_upload = files.upload()

for name in img_upload.keys():
    if name.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.move(name, 'inputs/pet_source.jpg')
        print("✅ 已保存为 inputs/pet_source.jpg")
        break


In [ ]:
print("📤 （可选）上传驱动音频 .wav，不上传则使用示例音频")
audio_upload = files.upload()

driven_audio = None
for name in audio_upload.keys():
    if name.lower().endswith('.wav'):
        shutil.move(name, 'inputs/driving.wav')
        driven_audio = 'inputs/driving.wav'
        print("✅ 已保存为 inputs/driving.wav")
        break

if driven_audio is None:
    print("⚠️ 未上传音频，将使用 SadTalker 自带示例音频")
    driven_audio = 'examples/driven_audio/sa1.wav'  # 仓库自带示例路径（按实际仓库结构调整）


## 步骤 4: 运行 SadTalker 生成宠物动画视频


In [ ]:
# 运行 SadTalker 推理

source_image = 'inputs/pet_source.jpg'
result_dir = 'results/pet_demo'
os.makedirs(result_dir, exist_ok=True)

print("🚀 开始生成宠物动画视频...")
!python inference.py \
  --driven_audio {driven_audio} \
  --source_image {source_image} \
  --result_dir {result_dir} \
  --still \
  --preprocess full \
  --enhancer none

print("\n📁 结果目录内容:")
!ls -lh {result_dir}


## 步骤 5: 从视频抽帧并生成 7×7 网格图


In [ ]:
# 工具函数：从视频抽帧并拼接网格
import cv2
import numpy as np
from PIL import Image
import math
import time
import json

def extract_frames_from_video(video_path, target_count=49):
    print(f"正在从视频提取帧: {video_path}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception(f"无法打开视频: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"  总帧数: {total_frames}, 目标帧数: {target_count}")
    images = []
    if total_frames <= 0:
        success = True
        while success:
            success, frame = cap.read()
            if success:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                images.append(img)
    else:
        indices = np.linspace(0, total_frames - 1, target_count, dtype=int)
        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                images.append(img)
    cap.release()
    if not images:
        raise Exception("未能提取任何帧")
    if len(images) > target_count:
        indices = np.linspace(0, len(images) - 1, target_count, dtype=int)
        images = [images[i] for i in indices]
    print(f"✅ 实际使用帧数: {len(images)}")
    return images

def create_grid_from_images(images, output_path='grid_pet_7x7.webp'):
    if not images:
        raise Exception("没有图片用于拼接")
    count = len(images)
    grid_size = math.ceil(math.sqrt(count))  # 49 -> 7
    cols = rows = grid_size
    w, h = images[0].size
    grid_w, grid_h = cols * w, rows * h
    grid_img = Image.new('RGB', (grid_w, grid_h))
    for idx, img in enumerate(images):
        if idx >= cols * rows:
            break
        c = idx % cols
        r = idx // cols
        grid_img.paste(img, (c * w, r * h))
    grid_img.save(output_path, 'WEBP', quality=85)
    meta = {
        'rows': rows,
        'cols': cols,
        'created_at': time.time()
    }
    with open(output_path.replace('.webp', '.json'), 'w') as f:
        json.dump(meta, f)
    print(f"✅ 网格图已保存: {output_path}")
    print(f"✅ 元数据已保存: {output_path.replace('.webp', '.json')}")
    return output_path, meta

print("✅ 抽帧与拼接函数已就绪")


In [ ]:
# 选择生成的视频（假设 SadTalker 将输出保存为 mp4）
import glob

video_candidates = glob.glob('results/pet_demo/*.mp4')
if not video_candidates:
    raise Exception('未找到生成的视频，请检查 SadTalker 输出目录')

video_path = video_candidates[0]
print(f"使用视频: {video_path}")

frames = extract_frames_from_video(video_path, target_count=49)
output_grid, meta = create_grid_from_images(frames, output_path='grid_pet_7x7.webp')


## 步骤 6: 下载网格图到本地


In [ ]:
from google.colab import files

print("📥 开始下载网格图和元数据...")
files.download('grid_pet_7x7.webp')
files.download('grid_pet_7x7.json')

print("✅ 下载完成，将两个文件放入本地 `face_demo/outputs/` 目录即可在前端使用")
